# Image Generator (Multi-class) — Fruits CNN


In [2]:

from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tqdm.auto import tqdm
import shutil
import os
import numpy as np

PROJECT_DIR = Path.cwd()

# Original dataset containing four class folders
ORIGIN_DIR = PROJECT_DIR / "dataset" / "origin"

# Generated dataset after splitting and augmentation
PREPARED_DIR = PROJECT_DIR / "dataset" / "prepared"

CLASS_NAMES = ["apple", "banana", "mango", "orange"]

IMAGE_SIZE = (128, 128)
VALIDATION_SIZE = 0.20
RANDOM_STATE = 42

# Number of augmented images per original image
AUGMENTATIONS_PER_IMAGE = 5

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

print("Project directory:", PROJECT_DIR)
print("Original dataset:", ORIGIN_DIR)
print("Prepared dataset:", PREPARED_DIR)

Project directory: C:\Users\fhv\rahat-codes\Multilevel Classification CNN Model Project
Original dataset: C:\Users\fhv\rahat-codes\Multilevel Classification CNN Model Project\dataset\origin
Prepared dataset: C:\Users\fhv\rahat-codes\Multilevel Classification CNN Model Project\dataset\prepared
apple     : 20 images
banana    : 13 images
mango     : 8 images
orange    : 14 images

Dataset directory verification completed.


## 1. Checking source folders and image counts


In [4]:
if not ORIGIN_DIR.exists():
    raise FileNotFoundError(
        f"Dataset directory not found: {ORIGIN_DIR}"
    )

for class_name in CLASS_NAMES:
    class_dir = ORIGIN_DIR / class_name

    if not class_dir.exists():
        raise FileNotFoundError(
            f"Missing class directory: {class_dir}"
        )

    image_files = [
        p for p in class_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]

    print(f"{class_name:10s}: {len(image_files)} images")

    if len(image_files) < 2:
        raise ValueError(
            f"Not enough images in {class_name}. "
            "Add more images before splitting."
        )

print("\nDataset directory verification completed.")

apple     : 20 images
banana    : 13 images
mango     : 8 images
orange    : 14 images

Dataset directory verification completed.


## 2. Split original training images into train and validation

We split the original training images by file path with stratification. This happens **before** augmentation, so augmented versions of one original image cannot leak into validation.

The official Fruits-360 `test` partition is not used in this split.

In [7]:
files, labels = [], []

for name in CLASS_NAMES:
    imgs = list_images(ORIGIN_DIR / name)
    files.extend(imgs)
    labels.extend([name] * len(imgs))

train_files, temp_files, train_labels, temp_labels = train_test_split(
    files, labels, test_size=0.30, random_state=RANDOM_STATE, stratify=labels
)

val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files, temp_labels, test_size=0.50,
    random_state=RANDOM_STATE, stratify=temp_labels
)

print("Train:", len(train_files), Counter(train_labels))
print("Validation:", len(val_files), Counter(val_labels))
print("Test:", len(test_files), Counter(test_labels))

Train: 38 Counter({'apple': 14, 'orange': 10, 'banana': 9, 'mango': 5})
Validation: 8 Counter({'apple': 3, 'orange': 2, 'banana': 2, 'mango': 1})
Test: 9 Counter({'apple': 3, 'banana': 2, 'orange': 2, 'mango': 2})


## 3. Dataset Checking

In [8]:
if PREPARED_DIR.exists():
    shutil.rmtree(PREPARED_DIR)

for split, files, labels in [
    ("train", train_files, train_labels),
    ("validation", val_files, val_labels),
    ("test", test_files, test_labels)
]:
    for src, name in zip(files, labels):
        dest = PREPARED_DIR / split / name
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest / src.name)

print("Prepared dataset created successfully!")

Prepared dataset created successfully!


## 4. Configure augmentation

The reference uses:
- `rescale=1./255`
- horizontal and vertical shifts
- nearest-pixel fill mode
- 30 saved variants per original image

The original reference's zoom and brightness ranges were `[1.0, 1.0]`, which do not actually vary zoom or brightness. Here we use modest non-identity ranges and a small rotation to create meaningful but realistic variation.

Augmentation is applied only to the training split.

In [9]:
augmentation_generator = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=15,
    zoom_range=0.15,
    brightness_range=(0.85, 1.15),
    height_shift_range=0.15,
    width_shift_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

print("Augmentations per original image:", AUGMENTATIONS_PER_IMAGE)
print("Training originals:", len(train_files))
print("Expected generated images:", len(train_files) * AUGMENTATIONS_PER_IMAGE)

Augmentations per original image: 5
Training originals: 38
Expected generated images: 190


## 5. Generate augmented training images

For each original training image, this cell saves the requested number of augmented PNG images in the same class folder. Validation and test images remain untouched.

If you need to rerun this cell, the notebook rebuild step above clears the previous generated output first.

In [10]:
generated_total, failed_images = 0, []

for name in CLASS_NAMES:
    folder = PREPARED_DIR / "train" / name
    originals = list_images(folder)
    count = 0

    for path in tqdm(originals, desc=f"Augmenting {name}"):
        try:
            image = img_to_array(load_img(path, target_size=IMAGE_SIZE))
            flow = augmentation_generator.flow(
                np.expand_dims(image, 0), batch_size=1,
                save_to_dir=folder, save_prefix=f"aug_{path.stem}",
                save_format="png"
            )

            for _ in range(AUGMENTATIONS_PER_IMAGE):
                next(flow)
                count += 1

        except Exception as e:
            failed_images.append((str(path), str(e)))

    generated_total += count
    print(f"{name}: {count} generated")

print(f"\nTotal generated: {generated_total}")
print(f"Failed images: {len(failed_images)}")

Augmenting apple:   0%|          | 0/14 [00:00<?, ?it/s]

apple: 70 generated


Augmenting banana:   0%|          | 0/9 [00:00<?, ?it/s]

banana: 45 generated


Augmenting mango:   0%|          | 0/5 [00:00<?, ?it/s]

mango: 25 generated


Augmenting orange:   0%|          | 0/10 [00:00<?, ?it/s]

orange: 50 generated

Total generated: 190
Failed images: 0


## 6. Verify final prepared counts

Training counts include original images plus augmented variants. Validation and test counts should equal the number of original files in their respective partitions.

In [11]:
def count_images(folder):
    return sum(p.suffix.lower() in IMAGE_EXTENSIONS for p in folder.rglob("*") if p.is_file())

for split in ["train", "validation", "test"]:
    counts = [count_images(PREPARED_DIR / split / name) for name in CLASS_NAMES]
    print(f"\n{split.upper()}:")
    for name, count in zip(CLASS_NAMES, counts):
        print(f"{name:10}: {count:6}")
    print(f"{'TOTAL':10}: {sum(counts):6}")

print("\nDataset:", PREPARED_DIR)


TRAIN:
apple     :     84
banana    :     54
mango     :     30
orange    :     60
TOTAL     :    228

VALIDATION:
apple     :      3
banana    :      2
mango     :      1
orange    :      2
TOTAL     :      8

TEST:
apple     :      3
banana    :      2
mango     :      2
orange    :      2
TOTAL     :      9

Dataset: C:\Users\fhv\rahat-codes\Multilevel Classification CNN Model Project\dataset\prepared


## 7. Final check and next step

Check that:
- All four classes contain images in all three splits.
- The training split contains originals and generated variants.
- Validation and test counts were not inflated by augmentation.
- `failed_images` is empty, or any failures have been investigated.

**Next notebook:** `Fit.ipynb` — it will load these folders using Keras directory generators and train the adapted CNN.